# 🔍 Exploration Descriptive des données de Sécurité (Couche Bronze)

Ce notebook est dédié à l'analyse descriptive brute des données de sécurité pour préparer les règles de gestion de la couche Silver.

**Objectif :** Comprendre la structure, la distribution et la qualité des données sans appliquer de transformations permanentes.

In [ ]:
import os
import sys

# Correction du problème d'import : ajout de la racine du projet au PYTHONPATH
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from pyspark.sql.functions import col, count, sum, min, max, desc
from src.common.spark_session_manager import get_spark_session
from src.config import SECURITE_BRONZE_PATH

# Initialisation Spark
spark = get_spark_session(app_name="Exploration_Descriptive_Securite")

# Activation de l'affichage interactif "Databricks-style"
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 50)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 200)

## 📥 1. Chargement des données Brutes (Bronze)
On observe ici la donnée telle qu'elle a été ingérée, sans aucun cast ni nettoyage.

In [ ]:
bronze_path = os.path.join(SECURITE_BRONZE_PATH, "security-filtered")
df_raw = spark.read.parquet(bronze_path)

print(f"Dimensions du dataset : {df_raw.count()} lignes x {len(df_raw.columns)} colonnes")
df_raw.limit(5)

## 📊 2. Analyse du Schéma et des Types
C'est ici que nous verrons les colonnes qui nécessiteront un cast (String vers Int/Double).

In [ ]:
df_raw.printSchema()

## 🔬 3. Statistiques Descriptives
Un aperçu rapide de la distribution des valeurs. 
*Note : Comme les types sont en String, Spark ne calculera pas la moyenne/stddev sur les nombres, mais cela nous montre les valeurs min/max textuelles.*

In [ ]:
df_raw.describe().limit(10)

## 🎯 4. Analyse des Indicateurs (Criminalité)
Quelles sont les thématiques présentes et leur volume ?

In [ ]:
df_raw.groupBy("indicateur") \
      .agg(count("*").alias("nb_occurences")) \
      .sort(desc("nb_occurences"))

## 🌍 5. Analyse Géographique et Temporelle
Vérification de l'exhaustivité des 9 arrondissements de Lyon et de la plage d'années.

In [ ]:
print("Répartition par Arrondissement (Code INSEE) :")
df_raw.groupBy("CODGEO_2025").count().sort("CODGEO_2025")

In [ ]:
print("Couverture Temporelle :")
df_raw.groupBy("annee").count().sort("annee")

## ❓ 6. Analyse des Valeurs Nulles (Qualité)
Identification des colonnes vides ou incomplètes.

In [ ]:
from pyspark.sql.functions import isnan, when, count, col

df_raw.select([count(when(col(c).isNull(), c)).alias(c) for c in df_raw.columns])

## 🔬 7. Zoom sur les Données Estimées (Redressement Statistique)
Certaines lignes contiennent des valeurs dans `complement_info_nombre` et `complement_info_taux`.
C'est la part du chiffre qui a été estimée par le SSMSI.

In [ ]:
from pyspark.sql.functions import desc

# Filtrage pour ne voir que les lignes de Lyon ayant des infos complémentaires
df_estimations = df_raw.filter(
    (col("CODGEO_2025").rlike("^6938[1-9]$")) & 
    (col("complement_info_nombre").isNotNull() | col("complement_info_taux").isNotNull())
)

print(f"Nombre de lignes avec estimations pour Lyon : {df_estimations.count()}")

# Affichage des colonnes clés pour comprendre le redressement
df_estimations.select(
    "annee", 
    "CODGEO_2025", 
    "indicateur", 
    "nombre", 
    "complement_info_nombre", 
    "taux_pour_mille", 
    "complement_info_taux"
).sort("annee", "CODGEO_2025")

## 👮 8. Corrélation entre Secret Statistique (ndiff) et Valeurs Manquantes
Vérifions ton hypothèse : est-ce que les `NULL` dans la colonne `nombre` sont systématiquement liés au statut `ndiff` (non diffusé) ?

In [ ]:
print("Répartition du statut 'est_diffuse' quand le NOMBRE est NULL :")
df_raw.filter(col("nombre").isNull()) \
      .groupBy("est_diffuse") \
      .count()

**Conclusion :** Si 100% des `NULL` sont en `ndiff`, cela confirme que le masquage est intentionnel pour respecter le secret statistique (très faible volume ou sensibilité).

## 🚀 9. Visualisation du résultat final (Couche Silver)

Cette cellule charge les données telles qu'elles ont été transformées par le script `src/etl/silver/silver_securite.py`.

In [ ]:
from src.config import SILVER_PATH
import os

# Chemin vers les données Silver traitées par le script .py
securite_silver_path = os.path.join(SILVER_PATH, "securite", "lyon_securite")

if os.path.exists(securite_silver_path):
    df_final = spark.read.parquet(securite_silver_path)
    print(f"✅ Données Silver chargées : {df_final.count()} lignes")
    df_final.sort("date_reference", "code_geo").limit(10)
else:
    print(f"❌ Le fichier Silver n'existe pas encore à l'emplacement : {securite_silver_path}")
    print("Pense à lancer le script : python -m src.etl.silver.silver_securite")

## 🚩 10. Audit de Qualité : Détection des Valeurs Nulles

Cette cellule identifie tous les enregistrements qui possèdent au moins une valeur `NULL` dans l'une de leurs colonnes.

In [ ]:
import functools
from pyspark.sql.functions import col

# Création d'un filtre dynamique pour toutes les colonnes
columns_to_check = df_final.columns
null_filter = functools.reduce(
    lambda a, b: a | b, 
    [col(c).isNull() for c in columns_to_check]
)

df_nulls = df_final.filter(null_filter)

print(f"⚠️ Nombre d'enregistrements avec au moins un NULL : {df_nulls.count()}")
if df_nulls.count() > 0:
    df_nulls.show(50, truncate=False)
else:
    print("✨ Qualité parfaite : Aucun NULL détecté dans le dataset Silver.")